# 4. Prever, na abertura, se um chamado será atendido em 30 dias

Quando alguém liga para o 156, a central não sabe dizer quanto tempo o pedido vai levar, e a supervisão só descobre que um chamado atrasou depois que ele atrasou. Um modelo que estime, no momento da abertura, a chance de atendimento em até 30 dias serve para duas coisas: informar ao cidadão um prazo realista e apontar cedo os chamados que precisam de atenção.

Hipótese H4: um modelo que usa bairro, data, chuva e demanda recente, além do serviço, prevê o atendimento em 30 dias melhor do que a simples taxa histórica de cada serviço.

O código fica em [`src/ml`](../src/ml); `make train` treina o modelo, salva o arquivo usado pelo dashboard e atualiza [`docs/ml_results.md`](../docs/ml_results.md).

## Por que esse alvo

A primeira ideia seria prever quantos dias o chamado leva. O problema é que 23% dos chamados ainda estão abertos e não têm tempo de resolução. Treinar só com os atendidos ensinaria ao modelo um mundo em que tudo é resolvido, e ele subestimaria justamente os serviços com fila grande. Prever "atendido em até 30 dias, sim ou não" resolve isso: um chamado aberto há mais de 30 dias e ainda na fila é um "não" conhecido. Só ficam de fora os chamados abertos nos últimos 30 dias, cujo desfecho ainda não se sabe.

In [1]:
import pandas as pd

from src.db.load import get_engine
from src.ml.features import build_target, load_model_data, outcome_is_known, temporal_split
from src.ml.train import fit_and_evaluate

tickets, rain = load_model_data(get_engine())
reference = tickets["status_updated_on"].max()
known = outcome_is_known(tickets, reference)
target = build_target(tickets)

by_group = (
    pd.DataFrame({"group": tickets["group_name"], "resolved_30d": target})[known]
    .groupby("group")["resolved_30d"]
    .agg(tickets="size", share_resolved_30d="mean")
    .nlargest(8, "tickets")
)
by_group.assign(share_resolved_30d=lambda df: (100 * df["share_resolved_30d"]).round(1))

,tickets,share_resolved_30d
group,,
ILUMINACAO PUBLICA,90892,97.2
LIMPEZA URBANA,39289,66.2
ARBORIZACAO,21812,30.8
DRENAGEM,17434,9.2
PAVIMENTACAO,8365,11.8
DENUNCIAS,4134,57.6
MANUT. URBANA,2510,0.1
FISCALIZACAO URBANA,2069,66.0


Dois terços dos chamados são atendidos em até 30 dias, mas a taxa muda completamente de um serviço para outro: quase todos os de iluminação, menos de um terço dos de drenagem e arborização. Isso já avisa que o tipo de serviço vai pesar muito, e que o modelo precisa ser comparado com uma régua honesta: a taxa histórica de cada serviço.

## Validação temporal

O modelo é treinado com chamados mais antigos e testado em meses posteriores, que ele nunca viu, como aconteceria no uso real. Entre treino e teste há um intervalo de 30 dias, para que nenhum rótulo de treino dependa do que aconteceu no período de teste. Nunca se usa divisão aleatória: ela misturaria passado e futuro e daria um resultado otimista demais.

In [2]:
split = temporal_split(reference)
run = fit_and_evaluate(tickets, rain, split)
print(f"Treino: abertos até {split.train_end:%d/%m/%Y}, {run['sizes']['train']} chamados")
test_period = f"{split.test_start:%d/%m/%Y} a {split.test_end:%d/%m/%Y}"
print(f"Teste: abertos de {test_period}, {run['sizes']['test']} chamados")
pd.DataFrame(run["results"]).T.round(3)

Treino: abertos até 31/05/2025, 146062 chamados
Teste: abertos de 01/07/2025 a 01/12/2025, 43670 chamados


,auc,brier,accuracy,alert_precision,alert_recall
Taxa geral do treino,0.500,0.222,0.667,0.000,0.000
Histórico do serviço,0.954,0.073,0.898,0.834,0.868
Gradient boosting,0.960,0.067,0.912,0.892,0.836


Contra a taxa geral qualquer coisa ganha. A régua que importa é o histórico do serviço, que já acerta muito (AUC 0,954) porque o tipo de serviço praticamente decide o prazo. O gradient boosting fica um pouco acima: AUC 0,960 e Brier 0,067 contra 0,073. No alerta de atraso (chance abaixo de 50%) ele é mais preciso, com 89% dos chamados apontados de fato passando de 30 dias, contra 83% do histórico, ao custo de um recall um pouco menor (84% contra 87%). H4 se confirma, mas o ganho geral é pequeno.

## O que o modelo usa

In [3]:
run["importances"].round(4).rename("queda_da_auc_ao_embaralhar").to_frame()

,queda_da_auc_ao_embaralhar
service_name,0.4582
neighborhood_name,0.0125
service_demand_7d,0.0015
group_name,0.0008
neighborhood_demand_30d,0.0004
rpa,0.0000
day_of_week,0.0000
is_weekend,0.0000
month,-0.0003
rain_3d_mm,-0.0004


Embaralhar o serviço derruba a AUC em 0,46: ele é quase tudo. Bem atrás vem o bairro, o que conversa com a desigualdade regional encontrada no notebook 02. A demanda recente do serviço ajuda um pouco; chuva, mês e dia da semana não acrescentam nada que serviço e bairro já não digam. Eles mudam o volume de chamados (notebooks 02 e 03), não o prazo de cada um.

## Onde o modelo ganha do histórico do serviço

In [4]:
test = run["test_frame"]
baseline = test["service_name"].map(run["service_rates"]).fillna(run["global_rate"])
errors = pd.DataFrame(
    {
        "group": test["group_name"],
        "baseline_error": (baseline - test["actual"]) ** 2,
        "model_error": (test["model_chance"] - test["actual"]) ** 2,
    }
)
top_groups = errors["group"].value_counts().head(5).index
(
    errors[errors["group"].isin(top_groups)]
    .groupby("group")[["baseline_error", "model_error"]]
    .mean()
    .assign(improvement_pct=lambda df: 100 * (1 - df["model_error"] / df["baseline_error"]))
    .round(3)
    .sort_values("improvement_pct", ascending=False)
)

,baseline_error,model_error,improvement_pct
group,,,
ARBORIZACAO,0.167,0.103,38.324
DRENAGEM,0.098,0.073,25.477
PAVIMENTACAO,0.061,0.056,8.332
LIMPEZA URBANA,0.135,0.140,-3.678
ILUMINACAO PUBLICA,0.031,0.032,-3.857


O ganho está onde o prazo é incerto: o erro cai 38% em arborização e 25% em drenagem, justamente os serviços em que a região muda a espera. Em iluminação e limpeza, já previsíveis pelo serviço, o modelo empata com o histórico ou fica levemente pior.

## As probabilidades são confiáveis?

In [5]:
bins = pd.cut(test["model_chance"], [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0], include_lowest=True)
test.groupby(bins, observed=True).agg(
    tickets=("actual", "size"),
    predicted=("model_chance", "mean"),
    observed=("actual", "mean"),
).round(3)

,tickets,predicted,observed
model_chance,,,
"(-0.001, 0.1]",9181,0.018,0.017
"(0.1, 0.3]",2142,0.191,0.254
"(0.3, 0.5]",2305,0.400,0.338
"(0.5, 0.7]",1409,0.601,0.517
"(0.7, 0.9]",3870,0.831,0.791
"(0.9, 1.0]",24763,0.983,0.964


As probabilidades batem com o observado nos extremos, onde estão quase todos os chamados: quando o modelo diz menos de 10%, menos de 2% são atendidos a tempo; quando diz mais de 90%, 96% são. Na faixa do meio ele é um pouco otimista (prevê 60%, acontecem 52%), mas ela concentra poucos chamados.

## Conclusão

H4 confirmada com ganho modesto: o histórico do serviço já explica quase tudo, e o modelo acrescenta o efeito do bairro, que vale sobretudo para drenagem e arborização. Na prática:

- para a central, a chance prevista é um prazo realista para informar ao cidadão;
- para a supervisão, a página Previsões do dashboard lista os chamados recentes com menor chance de atendimento a tempo, os candidatos a priorização.

Limitações:

- o tempo de atendimento é a data da última mudança de situação, que depende de como cada equipe registra o andamento;
- sem protocolo, a deduplicação entre arquivos anuais é aproximada (notebook 01);
- o modelo não conhece a capacidade das equipes, contratos ou mutirões, que provavelmente explicam boa parte da diferença entre regiões;
- treino e teste cobrem um ano e meio; como o comportamento das equipes muda, o modelo é retreinado a cada carga (`docker compose up` ou `make train`) e deve ser acompanhado com a mesma validação temporal.